In [ ]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4288 # the nearest multiple of 64 to 8/3*1600

In [21]:
num_params = 0
# Trainable parameters:
# input
# embedding
num_params += vocab_size * d_model # embedding table
# transformer_block_1
# ...
# transformer_block_{num_layers}
num_params += num_layers * (4*d_model**2 + 2*d_model + 3*d_model*d_ff)
# norm
num_params += d_model
# linear (output embedding)
num_params += d_model * vocab_size


In [22]:
print(f'total number of trainable parameters is: {num_params}')

total number of trainable parameters is: 1640452800


In [ ]:
from cs336_basics.model.transformer_lm import TransformerLanguageModel
model = TransformerLanguageModel(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, 1)

In [31]:


pytorch_total_params = sum(p.numel() for p in model.parameters())
pytorch_total_params

1640452800

In [32]:
model_bytes = num_params * 4 # 4 bytes per single-precision float

In [33]:
model_bytes / 1e9

6.5618112

In [88]:
flops_multihead_attention = num_layers * 4 * (2 * context_length * d_model**2)
flops_self_attention = num_layers*2*context_length**2*d_model
flops_ffn = num_layers*3*2*d_model*d_ff*context_length
flops_unembedding = 2*context_length*d_model*vocab_size

In [89]:
flops_multihead_attention / 1e9, flops_self_attention / 1e9, flops_ffn / 1e9, flops_unembedding / 1e9

(1006.63296, 161.0612736, 2023.3322496, 164.6821376)

In [100]:
def proportions(vocab_size, context_length, num_layers, d_model, num_head, d_ff, name):
    flops_multihead_attention = num_layers * 4 * (2 * context_length * d_model**2)
    flops_self_attention = num_layers*2*context_length**2*d_model
    flops_ffn = num_layers*3*2*d_model*d_ff*context_length
    flops_unembedding = 2*context_length*d_model*vocab_size
    total_flops = flops_unembedding + flops_ffn + flops_self_attention + flops_multihead_attention
    
    print(f"{name}: {flops_multihead_attention / total_flops:.2%}, {flops_self_attention / total_flops:.2%}, {flops_ffn / total_flops:.2%}, {flops_unembedding / total_flops:.2%}")


proportions(vocab_size, context_length, 12, 768, 12, d_ff, name='GPT-2 small') # GPT-2 small
proportions(vocab_size, context_length, 24, 1024, 16, d_ff, name='GPT-2 medium') # GPT-2 medium
proportions(vocab_size, context_length, 36, 1280, 20, d_ff, name='GPT-2 large') # GPT-2 large


GPT-2 small: 14.53%, 4.84%, 60.83%, 19.80%
GPT-2 medium: 20.40%, 5.10%, 64.07%, 10.43%
GPT-2 large: 25.09%, 5.02%, 63.05%, 6.84%


In [ ]:
# Non-embedding param count for a standard transformer
N = num_layers * (4 * d_model**2 + 2 * d_model * d_ff)
# For SwiGLU FFN, use 3 * d_model * d_ff instead of 2 * d_model * d_ff

forward_flops_approx = 2 * N * context_length  # per sequence
training_flops_approx = 6 * N * context_length  # per sequence